# télos MDLM: 50M Parameter Training Suite (Beta 1.5, 1.5)
This notebook executes the 50M parameter scaling study targeting 1:25, 1:35, 1:40, and 1:45 ratios.

### Pipeline Specification:
1. **50M 1:25**: Net2Net upscaled from 25M 1:25 checkpoint (`phase_b_25m_1to25_mlx`).
2. **50M 1:35, 1:40, 1:45**: Trained from scratch with Beta(1.5, 1.5) masking.

Memory GC and `mx.clear_cache()` are enforced to keep RAM usage strictly contained on Metal GPU.


In [ ]:
import os
import sys
import time
import gc
import yaml
import math
import io
from pathlib import Path
import numpy as np

# Ensure working directory is project root
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

import mlx.core as mx
import mlx.nn as nn
from telos.model.mlx_components import MLXTelosTransformer, load_upscaled_weights
from telos.training.trainer import TelosMLXTrainer
from telos.data.tokenizer import load_tokenizer

def run_training_step(config_path, upscaled_source=None, resume_from=None, resume_step=0):
    print("=" * 85)
    print("STARTING 50M RUN: " + str(config_path))
    print("=" * 85)
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    
    model = MLXTelosTransformer(**cfg["model"])
    model.set_dtype(mx.bfloat16)
    
    if upscaled_source:
        src_ckpt, src_cfg = upscaled_source
        print("  [Net2Net] Upscaling model weights from: " + str(src_ckpt))
        load_upscaled_weights(model, cfg["model"], src_ckpt, src_cfg)
    
    if resume_from:
        print(f'  [Resume] Loading weights from {resume_from}')
        model.load_weights(resume_from)
    
    trainer = TelosMLXTrainer(model, cfg)
    trainer.train(resume_step=resume_step)
    
    del model, trainer
    gc.collect()
    mx.clear_cache()
    print("FINISHED 50M RUN: " + str(config_path) + "\n")

def find_latest_checkpoint(ckpt_dir_pattern):
    base_dir = Path("checkpoints")
    matching = sorted(list(base_dir.glob(f"{ckpt_dir_pattern}*")), key=lambda p: p.stat().st_mtime, reverse=True)
    if not matching:
        raise FileNotFoundError(f"No checkpoint directory matching {ckpt_dir_pattern}")
    weights_path = matching[0] / "model.safetensors"
    if weights_path.exists(): return str(weights_path)
    npz_path = matching[0] / "model.npz"
    if npz_path.exists(): return str(npz_path)
    raise FileNotFoundError(f"No weight file in {matching[0]}")

# PIPELINE DEFINITION
start_time = time.time()

print("\n>>> Training 50M 1:35 (From Scratch) <<<")
run_training_step("configs/phase_b_50m_1to35_mlx.yaml", resume_from="checkpoints/phase_b_50m_1to35_mlx/checkpoint_step_5000.safetensors", resume_step=5000)

total_elapsed = (time.time() - start_time) / 3600.0
print("=" * 85)
print(f"ALL 50M TARGET RUNS COMPLETED SUCCESSFULLY IN {total_elapsed:.2f} HOURS!\n")
print("=" * 85)
